# Rare & Orphan Diseases - RAG & Article Generation

### Environment configuration

In [1]:

! pip install -r requirements.txt
! pip install google-cloud-aiplatform


! playwright install

  Using cached grpcio-1.67.1-cp313-cp313-macosx_10_13_universal2.whl.metadata (3.9 kB)
Using cached grpcio-1.67.1-cp313-cp313-macosx_10_13_universal2.whl (10.9 MB)
  Attempting uninstall: grpcio
    Found existing installation: grpcio 1.76.0
    Uninstalling grpcio-1.76.0:
      Successfully uninstalled grpcio-1.76.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-status 1.76.0 requires grpcio>=1.76.0, but you have grpcio 1.67.1 which is incompatible.

[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
  Using cached grpcio-1.76.0-cp313-cp313-macosx_11_0_universal2.whl.metadata (3.7 kB)
Using cached grpcio-1.76.0-cp313-cp313-macosx_11_0_universal2.whl (11.8 MB)
  Attempting uninstall: grpcio
    Found existing installation: grpcio 1.67.1
    Uninstalling grpcio-1.67.1:
      Successfully uninstal

In [5]:
# set the disease of interest


TOPIC = "Informative and Accurate Entry of Duchenne Muscular Dystrophy, describing the rare disease, its symptoms and suspected causes, and potential treatments or current efforts to find treatements or self treatment IF availabale."

In [4]:
# --- Vertex/Gemini LLM provider config ---------------------------------------
import os

# Choose your provider
LLM_PROVIDER = "vertex"   

# Vertex (Gemini) model + project config
# You can also set these in your environment.
VERTEX_MODEL    = os.getenv("VERTEX_MODEL", "gemini-1.5-pro-002")
VERTEX_PROJECT  = os.getenv("VERTEX_PROJECT", "GCP-PROJECT-ID")
VERTEX_LOCATION = os.getenv("VERTEX_LOCATION", "us-central1")

GEMINI_API_KEY  = os.getenv("GCP_API_KEY", "")


In [6]:
# configurations, be sure to load appropriate 
# 1.) SERPER_API_KEY
# 2.) LITELLM_API_KEY
# 3.) LITELLM_API_BASE
# 4.) VERTEX_API_KEY

import json
import os
from typing import List, Tuple
from google.cloud import aiplatform


import dspy
import httpx
from dotenv import load_dotenv
from tqdm import tqdm

from src.dataclass import RetrievedDocument, LiteratureSearchAgentResponse, LiteratureSearchAgentRequest
from src.encoder import Encoder
from src.literature_search import LiteratureSearchAgent
from src.lm import init_lm, LanguageModelProviderConfig, LanguageModelProvider, LiteLLMServerConfig
from src.retriever_agent.serper_rm import SerperRM
from src.rag import RagAgent
from src.dataclass import RagResponse, RagRequest

load_dotenv()


True

In [10]:

load_dotenv()
import json
import os
from dataclasses import dataclass, field, replace
from datetime import datetime
from typing import Any, Dict, List, Optional, Protocol



# Get GCP config from environment
project_id = os.getenv("GCP_PROJECT_ID") 
gcp_api_key = os.getenv("GCP_API_KEY") 

# ============================================================================
# Vertex AI Client
# ============================================================================

class VertexLLMClient:
    def __init__(self, project_id: Optional[str] = None, location: str = "us-central1", gcp_api_key: Optional[str] = None):

        # If service account file provided (and it's actually a file path), use it
        if gcp_api_key and os.path.exists(gcp_api_key):
            os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = gcp_api_key
            # Try to extract project_id from JSON if not provided
            if not project_id:
                try:
                    with open(gcp_api_key, 'r') as f:
                        project_id = json.load(f).get("project_id")
                except (json.JSONDecodeError, IOError):
                    pass
        # If gcp_api_key is set but not a file, ignore it (might be an API key string)
        # We'll use Application Default Credentials instead
        
        # Get project_id from environment if still not set
        if not project_id:
            project_id = os.environ.get("GCP_PROJECT_ID") or os.environ.get("GOOGLE_CLOUD_PROJECT")
        
        if not project_id:
            raise ValueError(
                "project_id is required. Set GCP_PROJECT_ID env var, provide it directly, "
                "or ensure service account JSON contains 'project_id'"
            )
        
        # Initialize Vertex AI (will use ADC if GOOGLE_APPLICATION_CREDENTIALS not set)
        aiplatform.init(project=project_id, location=location)
        self.project_id = project_id

    def generate(self, prompt: str, *, model: str, temperature: float, max_tokens: int, **kwargs: Any) -> str:
        is_gemini = "gemini" in model.lower()
        
        if is_gemini:
            try:
                from vertexai.preview.generative_models import GenerativeModel
            except ImportError:
                from vertexai.generative_models import GenerativeModel
            
            # Normalize Gemini model names for Vertex AI
            # Try different naming formats that Vertex AI might use
            model_variants = [model]  # Try original first
            
            # Support newer Gemini 2.5 and 2.0 models
            if "gemini-2.5-pro" in model.lower():
                model_variants = ["gemini-2.5-pro", "gemini-2.0-flash-001", "gemini-1.5-pro-002"]
            elif "gemini-2.5-flash" in model.lower():
                model_variants = ["gemini-2.5-flash", "gemini-2.0-flash-001", "gemini-1.5-flash-002"]
            elif "gemini-2.0-flash" in model.lower():
                model_variants = ["gemini-2.0-flash-001", "gemini-2.5-flash", "gemini-1.5-flash-002"]
            elif "gemini-1.5-pro" in model.lower():
                model_variants = ["gemini-1.5-pro-002", "gemini-1.5-pro", "gemini-2.0-flash-001", "gemini-pro"]
            elif "gemini-1.5-flash" in model.lower():
                model_variants = ["gemini-1.5-flash-002", "gemini-1.5-flash", "gemini-2.0-flash-001", "gemini-flash"]
            elif "gemini-pro" in model.lower() and "2" not in model.lower():
                model_variants = ["gemini-pro", "gemini-2.0-flash-001", "gemini-1.5-pro-002"]
            
            config = {"temperature": temperature, "max_output_tokens": max_tokens}
            if kwargs.get("reasoning_effort"):
                config["reasoning_effort"] = kwargs["reasoning_effort"]
            
            # Try each model variant until one works
            last_error = None
            for variant in model_variants:
                try:
                    return GenerativeModel(variant).generate_content(prompt, generation_config=config).text
                except Exception as e:
                    last_error = e
                    if "404" not in str(e) and "not found" not in str(e).lower():
                        # Not a model not found error, re-raise
                        raise
                    continue
            
            # If all variants failed
            raise RuntimeError(
                f"None of the Gemini model variants {model_variants} are available. "
                f"Last error: {last_error}. "
                f"Try using 'text-bison@002' instead, or check available models in GCP Console."
            ) from last_error
        else:
            model_obj = aiplatform.TextGenerationModel.from_pretrained(model)
            params = {"temperature": temperature, "max_output_tokens": max_tokens}
            if kwargs.get("reasoning_effort"):
                params["reasoning_effort"] = kwargs["reasoning_effort"]
            
            response = model_obj.predict(prompt, **params)
            return response.text if hasattr(response, "text") else str(response)


def create_vertex_client(project_id: Optional[str] = None, location: str = "us-central1", gcp_api_key: Optional[str] = None) -> VertexLLMClient:
    return VertexLLMClient(project_id=project_id, location=location, gcp_api_key=gcp_api_key)

# Create Vertex client
test_lm = create_vertex_client(project_id=project_id, location=VERTEX_LOCATION, gcp_api_key=gcp_api_key)

# Test with a simple prompt
response = test_lm.generate(
    prompt="say 'Hello!' as is",
    model=VERTEX_MODEL,
    temperature=0.0,
    max_tokens=10
)
print(response) 

/Users/nnataliewang19/Documents/coterm q/fall cs 224v/conv-rare-disease/venv/lib/python3.13/site-packages/vertexai/generative_models/_generative_models.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()
/Users/nnataliewang19/Documents/coterm q/fall cs 224v/conv-rare-disease/venv/lib/python3.13/site-packages/vertexai/generative_models/_generative_models.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


Hello!






### Objective
Evaluate the complete Retrieval-Augmented Generation pipeline to understand information flow through the following stages:

```
Query → Internet Search → Content Extraction → Document Chunking → Semantic Reranking → Answer Generation
```

---

In [17]:
## Wikipedia Discovery & Guideline Generation

from guideline import find_wikipedia_articles, extract_wikipedia_info, Guideline

# Extract disease name from TOPIC
disease_name = "Duchenne Muscular Dystrophy"

# Simple LLM config object
class LLMConfig:
    def __init__(self):
        self.model_name = VERTEX_MODEL
        self.temperature = 0.4
        self.max_tokens = 8192

llm_config = LLMConfig()

# Helper functions for iterative refinement
def is_satisfied(articles, guideline, llm_client, llm_config, min_articles=3):
    """Check if current searches and guideline are satisfactory using multiple criteria."""
    if len(articles) < min_articles:
        return False, "Not enough articles found"
    
    # Check article diversity - look for key topics
    article_texts = " ".join([a.get('title', '') + " " + a.get('reason', '') for a in articles])
    article_texts_lower = article_texts.lower()
    
    # Key coverage areas for rare disease articles
    key_areas = {
        'main_disease': any(term in article_texts_lower for term in [disease_name.lower(), 'disease', 'disorder', 'syndrome']),
        'symptoms': any(term in article_texts_lower for term in ['symptom', 'sign', 'clinical', 'manifestation']),
        'treatment': any(term in article_texts_lower for term in ['treatment', 'therapy', 'drug', 'medication', 'intervention']),
        'genetics': any(term in article_texts_lower for term in ['genetic', 'gene', 'mutation', 'inheritance', 'chromosome']),
        'diagnosis': any(term in article_texts_lower for term in ['diagnosis', 'diagnostic', 'test', 'testing', 'detection'])
    }
    
    coverage_score = sum(key_areas.values()) / len(key_areas)
    
    # If we have good coverage, use LLM to evaluate guideline quality
    if coverage_score >= 0.6 and guideline is not None:
        prompt = f"""Evaluate if this guideline for "{disease_name}" is comprehensive enough to generate a research plan.

Guideline sections: {len(guideline.sections)}
Heuristics: {len(guideline.heuristics)}
Citation resources: {len(guideline.citation_resources)}

Key areas covered in Wikipedia articles:
{', '.join([k for k, v in key_areas.items() if v])}

Respond with ONLY "YES" if the guideline is comprehensive enough, or "NO" with a brief reason if it needs more information."""
        
        evaluation = llm_client.generate(
            prompt.strip(),
            model=llm_config.model_name,
            temperature=0.2,
            max_tokens=100
        ).strip()
        
        if evaluation.upper().startswith("YES"):
            return True, f"Comprehensive coverage ({coverage_score:.1%}) and LLM-approved"
        else:
            reason = evaluation.replace("NO", "").strip() if "NO" in evaluation.upper() else "Needs more information"
            return False, f"Coverage: {coverage_score:.1%}, {reason}"
    
    # Fallback: check coverage score
    if coverage_score >= 0.7:
        return True, f"Good coverage ({coverage_score:.1%})"
    else:
        missing = [k for k, v in key_areas.items() if not v]
        return False, f"Coverage: {coverage_score:.1%}, missing: {', '.join(missing)}"

def generate_next_search_query(disease_name, current_articles, guideline, llm_client, llm_config):
    """Generate next search query to improve guideline coverage."""
    current_titles = [a.get('title', '') for a in current_articles]
    current_titles_str = ", ".join(current_titles[:5])
    
    prompt = f"""Given the disease "{disease_name}" and current Wikipedia articles found:
{current_titles_str}

Generate a focused search query to find additional Wikipedia articles that would help improve guideline coverage.
Focus on gaps in: treatments, genetics, patient resources, related conditions, or research.

Return only a single search query string (no JSON, no explanation)."""
    
    query = llm_client.generate(
        prompt.strip(),
        model=llm_config.model_name,
        temperature=0.5,
        max_tokens=50
    ).strip()
    
    # Clean up the query (remove quotes if present)
    query = query.strip('"').strip("'").strip()
    return query

# Iterative refinement loop
all_articles = []
wikipedia_info = ""
guideline = None
max_iterations = 3
iteration = 0

print(f"🔍 Starting Wikipedia discovery for: {disease_name}\n")

while iteration < max_iterations:
    iteration += 1
    print(f"--- Iteration {iteration} ---")
    
    # Generate search query
    if iteration == 1:
        # First iteration: use disease name directly
        search_query = disease_name
    else:
        # Subsequent iterations: generate focused query
        search_query = generate_next_search_query(disease_name, all_articles, guideline, test_lm, llm_config)
        print(f"🔎 Generated search query: {search_query}")
    
    # Find Wikipedia articles
    new_articles = find_wikipedia_articles(search_query, test_lm, llm_config)
    
    # Deduplicate articles by title
    existing_titles = {a.get('title', '').lower() for a in all_articles}
    unique_new = [a for a in new_articles if a.get('title', '').lower() not in existing_titles]
    all_articles.extend(unique_new)
    
    print(f"✅ Found {len(unique_new)} new articles (total: {len(all_articles)})")
    for article in unique_new[:3]:  # Show first 3
        print(f"  - {article.get('title', 'Unknown')}: {article.get('reason', '')}")
    
    # Extract information from all articles
    print(f"\n📝 Extracting information from Wikipedia articles...")
    wikipedia_info = extract_wikipedia_info(all_articles, disease_name, test_lm, llm_config)
    
    # Generate guideline using LLM
    prompt = f"""Generate a comprehensive guideline for writing a Wikipedia-style article about "{disease_name}".

Style: patient_facing (for educated lay readers at 9th-10th grade reading level)

Wikipedia information gathered:
{wikipedia_info if wikipedia_info else "No Wikipedia information available yet."}

Create a guideline with:
1. Required sections (list of section names)
2. Writing heuristics (rules for writing style)
3. Citation resources (priority order)
4. Citation rules (how to cite)
5. Quality checklist

Return as JSON with keys: sections (list), heuristics (list), citation_resources (list), citation_rules (list), quality_checklist (list), style_notes (string), audience (string).

Example format:
{{
  "sections": ["Quick Facts", "Signs and Symptoms", ...],
  "heuristics": ["Use plain language", "Define technical terms", ...],
  "citation_resources": ["Peer-reviewed reviews", "NIH resources", ...],
  "citation_rules": ["Use [1], [2] format", "Cite every claim", ...],
  "quality_checklist": ["All claims cited", "Technical terms defined", ...],
  "style_notes": "Plain, respectful language at high school level...",
  "audience": "Informed layperson at 9th-10th grade reading level"
}}"""
    
    response = test_lm.generate(
        prompt.strip(),
        model=llm_config.model_name,
        temperature=0.4,
        max_tokens=2000
    )
    
    # Parse JSON response
    import json
    cleaned = response.strip()
    if cleaned.startswith("```"):
        lines = cleaned.split("\n")
        if len(lines) > 1:
            cleaned = "\n".join(lines[1:-1]) if lines[-1].strip() == "```" else "\n".join(lines[1:])
    
    try:
        guideline_dict = json.loads(cleaned)
        guideline = Guideline(
            disease_name=disease_name,
            style="patient_facing",
            audience=guideline_dict.get("audience", "Informed layperson at 9th-10th grade reading level"),
            sections=guideline_dict.get("sections", []),
            heuristics=guideline_dict.get("heuristics", []),
            citation_resources=guideline_dict.get("citation_resources", []),
            citation_rules=guideline_dict.get("citation_rules", []),
            quality_checklist=guideline_dict.get("quality_checklist", []),
            style_notes=guideline_dict.get("style_notes", "")
        )
    except (json.JSONDecodeError, KeyError) as e:
        print(f"⚠️  Error parsing guideline JSON: {e}")
        print(f"Response: {response[:200]}...")
        # Fallback: create minimal guideline
        guideline = Guideline(
            disease_name=disease_name,
            style="patient_facing",
            audience="Informed layperson at 9th-10th grade reading level",
            sections=["Quick Facts", "Signs and Symptoms", "Treatment", "References"],
            heuristics=["Use plain language", "Cite all claims"],
            citation_resources=["Peer-reviewed sources"],
            citation_rules=["Use numeric citations"],
            quality_checklist=["All claims cited"]
        )
    
    # Check if satisfied
    satisfied, reason = is_satisfied(all_articles, guideline, test_lm, llm_config)
    print(f"\n📊 Satisfaction check: {reason}")
    
    if satisfied:
        print(f"✅ Satisfied with {len(all_articles)} articles")
        break
    else:
        print(f"⚠️  Not satisfied yet, continuing search...")

# Save final guideline
import json
from pathlib import Path
from datetime import datetime

# Create output directory
output_dir = Path("output")
output_dir.mkdir(exist_ok=True)

# Generate filename with timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
disease_safe = disease_name.replace(" ", "_").replace("/", "_").lower()
guideline_json_path = output_dir / f"guideline_{disease_safe}_{timestamp}.json"
guideline_md_path = output_dir / f"guideline_{disease_safe}_{timestamp}.md"

# Save as JSON
with open(guideline_json_path, 'w', encoding='utf-8') as f:
    json.dump({
        "guideline": guideline.to_dict(),
        "metadata": {
            "disease_name": disease_name,
            "articles_found": len(all_articles),
            "articles": all_articles,
            "wikipedia_info": wikipedia_info,
            "iterations": iteration,
            "timestamp": timestamp
        }
    }, f, indent=2, ensure_ascii=False)

# Save as Markdown
with open(guideline_md_path, 'w', encoding='utf-8') as f:
    f.write(f"# Guideline for: {disease_name}\n\n")
    f.write(f"**Generated:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"**Articles found:** {len(all_articles)}\n")
    f.write(f"**Iterations:** {iteration}\n\n")
    f.write("---\n\n")
    f.write(str(guideline))
    f.write("\n\n---\n\n")
    f.write("## Wikipedia Articles Used\n\n")
    for i, article in enumerate(all_articles, 1):
        f.write(f"{i}. **{article.get('title', 'Unknown')}**\n")
        f.write(f"   - URL: {article.get('url', 'N/A')}\n")
        f.write(f"   - Reason: {article.get('reason', 'N/A')}\n\n")

print(f"\n Guideline generated!")
print(f"\n📄 Saved to:")
print(f"   - JSON: {guideline_json_path}")
print(f"   - Markdown: {guideline_md_path}")
print(f"\n{guideline}")


🔍 Starting Wikipedia discovery for: Duchenne Muscular Dystrophy

--- Iteration 1 ---
✅ Found 8 new articles (total: 8)
  - Duchenne muscular dystrophy: Main article about the disease, providing comprehensive information.
  - Becker muscular dystrophy: Related condition caused by mutations in the same gene (dystrophin), but with a milder phenotype.
  - Muscular dystrophy: Provides a broader overview of the muscular dystrophy family of diseases, placing Duchenne in context.

📝 Extracting information from Wikipedia articles...

📊 Satisfaction check: Coverage: 80.0%, . Citation resources are too low for a comprehensive research plan.
⚠️  Not satisfied yet, continuing search...
--- Iteration 2 ---
🔎 Generated search query: Duchenne muscular dystrophy" AND ("clinical trials" OR "emerging therapies" OR "genetic counseling" OR "patient advocacy" OR "care standards" OR "comorbidities" OR "cardiac involvement" OR "respiratory management
✅ Found 7 new articles (total: 15)
  - Clinical trial: Gene

In [61]:
# configure the language model for RAG agent
rag_lm_config = LanguageModelProviderConfig(
    provider=LanguageModelProvider.LANGUAGE_MODEL_PROVIDER_LITELLM_SERVER,
    model_name="gpt-4.1-mini",
    temperature=1.0,
    max_tokens=10000,
    litellm_server_config=LiteLLMServerConfig(api_key=os.getenv("LITELLM_API_KEY"), api_base=os.getenv("LITELLM_API_BASE"))
)
rag_lm = init_lm(rag_lm_config)

# initialize the RAG agent
rag = RagAgent(retriever=serper_retriever, rag_lm=rag_lm)

# forward the request to the RAG agent
rag_response: RagResponse = await rag.aforward(RagRequest(question="Provide a informative entry about Duchenne Muscular Dystrophy.", max_retriever_calls=1))

# make output directory if it doesn't exist
os.makedirs("output", exist_ok=True)

# save the response to a file
with open("output/action_item_3_rag_response.json", "w") as f:
    json.dump(rag_response.to_dict(), f, indent=2)

print(f"✅ Result saved to output/action_item_3_rag_response.json")


python3.11(42192) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


[FETCH]... ↓ https://my.clevelandclinic.org/health/diseases/23538-duchenne-muscular-dystrophy-dmd                 |
✓ | ⏱: 3.02s 

[SCRAPE].. ◆ https://my.clevelandclinic.org/health/diseases/23538-duchenne-muscular-dystrophy-dmd                 |
✓ | ⏱: 0.07s 

[COMPLETE] ● https://my.clevelandclinic.org/health/diseases/23538-duchenne-muscular-dystrophy-dmd                 |
✓ | ⏱: 3.10s 

[FETCH]... ↓ https://www.rch.org.au/kidsinfo/fact_sheets/DMD_information_for_carriers/                            |
✓ | ⏱: 3.27s 

...

[COMPLETE] ● https://www.ncbi.nlm.nih.gov/books/NBK482346/                                                        |
✓ | ⏱: 8.16s 

[FETCH]... ↓ https://www.parentprojectmd.org/about-duchenne/what-is-duchenne/                                     |
✓ | ⏱: 8.33s 

[SCRAPE].. ◆ https://www.parentprojectmd.org/about-duchenne/what-is-duchenne/                                     |
✓ | ⏱: 0.07s 

[COMPLETE] ● https://www.parentprojectmd.org/about-duchenne/what-is-duchenne/                                     |
✓ | ⏱: 8.40s 

✅ Result saved to output/action_item_3_rag_response.json


### Autonomous Literature Search Evaluation


---

In [33]:
literature_search_planning_lm = init_lm(LanguageModelProviderConfig(
    provider=LanguageModelProvider.LANGUAGE_MODEL_PROVIDER_LITELLM_SERVER,
    model_name="gpt-4.1", # NOTE: planning invovles intelligence, so we use a more powerful model.
    temperature=1.0,
    max_tokens=10000,
    litellm_server_config=LiteLLMServerConfig(api_key=os.getenv("LITELLM_API_KEY"), api_base=os.getenv("LITELLM_API_BASE"))
))

answer_synthesis_lm = init_lm(LanguageModelProviderConfig(
    provider=LanguageModelProvider.LANGUAGE_MODEL_PROVIDER_LITELLM_SERVER,
    model_name="gpt-5-mini", # NOTE: synthesis does not require high intelligence, but requires minimal hallucination. GPT-5-mini is a good balance.
    temperature=1.0,
    
    max_tokens=20000,
    litellm_server_config=LiteLLMServerConfig(api_key=os.getenv("LITELLM_API_KEY"), api_base=os.getenv("LITELLM_API_BASE"))
))

# initialize the literature search agent
literature_search_agent = LiteratureSearchAgent(rag_agent=rag, literature_search_lm=literature_search_planning_lm, answer_synthesis_lm=answer_synthesis_lm)

# run the literature search agent
literature_search_response: LiteratureSearchAgentResponse = await literature_search_agent.aforward(LiteratureSearchAgentRequest(topic=TOPIC))

# save the response to a file
with open("output/action_item_4_literature_search_response.json", "w") as f:
    json.dump(literature_search_response.to_dict(), f, indent=2)

print(f"✅ Result saved to output/literature_search_response.json")

Starting literature search for topic: Informative and Accurate Entry of Duchenne Muscular Dystrophy, describing the rare disease, its symptoms and suspected causes, and potential treatments or current efforts to find treatements or self treatment IF availabale.
Completeness check start.
Completeness check: False, reasoning: No major areas have been explored yet; foundational questions on description, symptoms, causes, and mechanisms of Duchenne Muscular Dystrophy are critical for initial comprehensive coverage.
Generated 3 next questions for exploration
Executing 3
RAG call start. Question: What is Duchenne Muscular Dystrophy (DMD), and how is it classified among rare diseases?. Question context: Establishing a clear and informative description of DMD, including its rarity and disease classification, is foundational to the survey and addresses the topic's requirement for an overview.
RAG call start. Question: What are the primary and secondary symptoms of Duchenne Muscular Dystrophy, a

python3.11(41492) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python3.11(41493) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python3.11(41494) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python3.11(41495) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python3.11(41496) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python3.11(41497) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


[FETCH]... ↓ https://www.parentprojectmd.org/about-duchenne/what-is-duchenne/progression/                         |
✓ | ⏱: 16.62s 

[SCRAPE].. ◆ https://www.parentprojectmd.org/about-duchenne/what-is-duchenne/progression/                         |
✓ | ⏱: 0.29s 

[COMPLETE] ● https://www.parentprojectmd.org/about-duchenne/what-is-duchenne/progression/                         |
✓ | ⏱: 16.94s 

...

[COMPLETE] ● https://publications.aap.org/pediatrics/article/...1542/Psychosocial-Management-of-the-Patient-With  |
✓ | ⏱: 118.46s 

[FETCH]... ↓ https://www.neurology.org/doi/10.1212/WNL.0000000000213604                                           |
✓ | ⏱: 119.75s 

[SCRAPE].. ◆ https://www.neurology.org/doi/10.1212/WNL.0000000000213604                                           |
✓ | ⏱: 0.66s 

[COMPLETE] ● https://www.neurology.org/doi/10.1212/WNL.0000000000213604                                           |
✓ | ⏱: 120.41s 

Completed iteration 10, remaining budget: 5
Completeness check start.
Completeness check: True, reasoning: All major dimensions essential for an informative and accurate entry about Duchenne Muscular Dystrophy—including disease description, classification, symptoms, suspected causes, potential/current treatments, experimental therapies, and recommended supportive/self-management strategies—have been explored in sufficient depth; further information gain within the stated topic and guideline is now low.
Literature search deemed complete by completeness checker
Survey completed with 5 responses
Starting final synthesis
Final synthesis complete
✅ Result saved to output/literature_search_response.json


In [68]:
guideline_large = '''
# {Disease name} ({Plain-language name})

{One-sentence summary in plain language} [n]

**Important safety note:** This article is for education only and does not replace advice from your clinician. If you have urgent symptoms, call emergency services.

## 1. Quick Facts
- Prevalence/incidence: {value, context} [n]
- Typical age of onset: {value} [n]
- Main symptoms: {3–6 bullets} [n]
- Diagnosis: {core test/criteria} [n]
- Cause/inheritance: {genetic/acquired; pattern if known} [n]
- Treatment approach: {categories only} [n]
- Prognosis: {plain summary} [n]

## 2. Names and Classification
- Synonyms: {list}
- IDs: {OMIM/Orphanet if available}
- Classification: {category} [n]

## 3. What Happens in the Body
{2–3 short paragraphs explaining mechanism and linking to symptoms} [n]

## 4. Signs and Symptoms
- Common: {bullets} [n]
- Less common: {bullets} [n]
- Red flags needing urgent care: {bullets} [n]

## 5. How It Is Diagnosed
- Clinical evaluation: {summary} [n]
- Key tests and what results mean: {bullets} [n]
- Differential diagnosis: {3–5 items with one-line distinctions} [n]

## 6. How Common Is It?
{short paragraph with numbers, geography, time period, confidence} [n]

## 7. Causes and Genetics
{genetic variants, inheritance, non-genetic causes, strength of evidence} [n]

## 8. Treatment and Management (Education-only overview)
### Medications
{approved therapies with mechanism and evidence summary} [n]
### Procedures/Devices
{if applicable} [n]
### Supportive Care
{common strategies reported in guidelines} [n]
### Monitoring
{what clinicians often follow} [n]
### Special Populations
{pediatric/pregnancy if supported} [n]

## 9. Prognosis and Living With the Condition
{natural history and quality-of-life evidence} [n]

## 10. Research and Clinical Trials
{current directions and how trials are typically found} [n]

## 11. Patient Resources
- {Resource name – what it offers}
- {Resource name – what it offers}

## 12. What Is Not Known
- {gap 1 with citation if available}
- {gap 2}
- {gap 3}
- {gap 4}

## 13. References
{[1] … will be auto-rendered elsewhere}
'''

## DSPy Implementation for Automated Thesis Generation

> **Reference:** See **Section 4.3 (DSPy Framework)** in the handout.
---

In [10]:
# class ThesisGenerator(dspy.Signature):
#     """
#     You are GENERATING exactly FIVE (5) INVESTIGATIVE thesis statements that synthesize 
#     THE ORIGINAL RESEARCH TOPIC, STRUCTURED DATABASE INSIGHTS, and LITERATURE REVIEW HIGHLIGHTS AND GAPS.

#     THe requrements for each thesis are as follows: Specific! Names teh actors and actions, avoids vagueness. Evidence based, CITE at least one concrete data point OR pattern from the 
#     database inputs OR precise claims from literature, Analytcically defensable, AND goes over societal impact EXPLICTLY must be CONCISE and MEMORABLE.

    

    
#     """
#     original_topic: str = dspy.InputField(
#         desc="The initial research topic that was explored"
#     ) 
#     # no need for DB analysis, we solely want literature search!
#     # db_figures: str = dspy.InputField(
#     #     desc="Responses from database exploration"
   
#     # )

    
#     # ===============================================
#     # additional input fields here
#     # Hint1: how to pass insights from database exploration agent to the thesis generator?
#     # Hint2: will directly concatenate question and answers from database exploration agent work? Do we need any formatting?
#     # ===============================================
    
#     # Should only have one output field as defined below. Do not change the name of the output field.
#     proposed_theses: List[str] = dspy.OutputField(
#         desc="""A python list of exactly five distinct, concise thesis. each is specific 
#         names actors and cites a conrete pattern or figure when critical. (specific, evidence-based, and analytically defensible)"""
#     )

# thesis_generator = dspy.Predict(ThesisGenerator)
# thesis_generator_lm = init_lm(LanguageModelProviderConfig(
#     provider=LanguageModelProvider.LANGUAGE_MODEL_PROVIDER_LITELLM_SERVER,
#     model_name="gpt-4.1", # NOTE: thesis generation requires high intelligence, so we use a more powerful model.
#     temperature=1.0,
#     max_tokens=10000,
#     litellm_server_config=LiteLLMServerConfig(api_key=os.getenv("LITELLM_API_KEY"), api_base=os.getenv("LITELLM_API_BASE"))
# ))

In [11]:
# database_exploration_rag_responses = []
# with open("data/action_item_5_database_exploration_precomputed.json", "r") as f:
#     database_exploration_precomputed = json.load(f)
#     database_exploration_rag_responses = database_exploration_precomputed.get("rag_service_responses", [])


# with dspy.context(lm=thesis_generator_lm):
#     generated_theses = (await thesis_generator.aforward(
#         original_topic=TOPIC,
#         # ===============================================
#         # other input fields here
#         # ===============================================
#         # db_figures = database_exploration_rag_responses, !
        
#     )).proposed_theses

# formatted_generated_theses = "\n\t-".join(generated_theses)
# print(f"generated_theses:\n\t- {formatted_generated_theses}")

# with open("output/action_item_6_generated_theses.json", "w") as f:
#     json.dump({
#         "generated_theses": generated_theses
#     }, f, indent=2)
# print(f"✅ Result saved to output/action_item_6_generated_theses.json")

# print(f"\n\nTake a look at the output format and skim through the content. Are you satisfied with the results? Will audience be interested in your proposed theses? With doubt, it's always a good idea to revise the prompt and try again.")
# if len(generated_theses) != 5:
#     raise ValueError(f"❌ Please generate exactly 5 proposed theses but found {len(generated_theses)}.")


generated_theses:
	- Despite being the most common and severe form of pediatric muscular dystrophy, Duchenne Muscular Dystrophy (DMD) is underdiagnosed in low-resource settings, with studies suggesting global diagnostic delays averaging 2.5 years after symptom onset—heightening the societal burden on affected families.
	-Exon-skipping therapies, pioneered by Sarepta Therapeutics and evidenced by FDA approval of eteplirsen in 2016, demonstrate a paradigm shift in DMD treatment, offering measurable improvements in ambulation yet still only benefiting a subset of patients with specific genetic mutations.
	-The lack of accessible genetic testing infrastructure in over 40% of countries globally hinders early detection of DMD, worsening disease outcomes and exacerbating inequity in care for marginalized populations.
	-Patient advocacy groups such as Parent Project Muscular Dystrophy have catalyzed critical increases in research funding, tripling US-based DMD therapeutic pipelines between 201

In [12]:
# # database_exploration_rag_responses = []
# with open("data/action_item_5_database_exploration_precomputed.json", "r") as f:
#     database_exploration_precomputed = json.load(f)
#     database_exploration_rag_responses = ... # TODO: add the rag responses to the list

# with dspy.context(lm=thesis_generator_lm):
#     generated_theses = (await thesis_generator.aforward(
#         original_topic=TOPIC,
#         # ===============================================
#         # other input fields here
#         # ===============================================
#     )).proposed_theses

# formatted_generated_theses = "\n\t-".join(generated_theses)
# print(f"generated_theses:\n\t- {formatted_generated_theses}")

# with open("output/action_item_6_generated_theses.json", "w") as f:
#     json.dump({
#         "generated_theses": generated_theses
#     }, f, indent=2)
# print(f"✅ Result saved to output/action_item_6_generated_theses.json")

# print(f"\n\nTake a look at the output format and skim through the content. Are you satisfied with the results? Will audience be interested in your proposed theses? With doubt, it's always a good idea to revise the prompt and try again.")
# if len(generated_theses) != 5:
#     raise ValueError(f"❌ Please generate exactly 5 proposed theses but found {len(generated_theses)}.")


generated_theses:
	- Despite being the most common and severe form of pediatric muscular dystrophy, Duchenne Muscular Dystrophy (DMD) is underdiagnosed in low-resource settings, with studies suggesting global diagnostic delays averaging 2.5 years after symptom onset—heightening the societal burden on affected families.
	-Exon-skipping therapies, pioneered by Sarepta Therapeutics and evidenced by FDA approval of eteplirsen in 2016, demonstrate a paradigm shift in DMD treatment, offering measurable improvements in ambulation yet still only benefiting a subset of patients with specific genetic mutations.
	-The lack of accessible genetic testing infrastructure in over 40% of countries globally hinders early detection of DMD, worsening disease outcomes and exacerbating inequity in care for marginalized populations.
	-Patient advocacy groups such as Parent Project Muscular Dystrophy have catalyzed critical increases in research funding, tripling US-based DMD therapeutic pipelines between 201

**Extra human-in-the-loop step**
Manually review all generated thesis and hand pick the best 2

In [69]:
# # Review the generated these and hand pick the best 2 theses.
# selected_theses = [
#     "Genetic mutations in the DMD gene are confirmed as the singular cause of Duchenne Muscular Dystrophy, as evidenced by over 95% of diagnosed cases showing deletions or duplications in this locus, underscoring the necessity for molecular diagnostic protocols in clinical settings (Emery, 2023).",
#     "Progressive muscle wasting and cardiac involvement, observed consistently in longitudinal studies of DMD patients (Mendell et al., 2016), directly correlate with reduced life expectancy—highlighting an urgent societal need for expanded cardiopulmonary care in pediatric neuromuscular clinics."
# ]

# with open("output/action_item_6_selected_theses.json", "w") as f:
#     json.dump({
#         "selected_theses": selected_theses
#     }, f, indent=2)
# print(f"✅ Result saved to output/action_item_6_selected_theses.json")

✅ Result saved to output/action_item_6_selected_theses.json


## Automated Investigative Report Synthesis

---

In [74]:
# --- Use ./outputs as the artifacts root -------------------------------------
import os, json
from pathlib import Path
from datetime import datetime

# Force artifacts into ./outputs
ARTIFACT_ROOT = Path("outputs")

def _is_writable_dir(p: Path) -> bool:
    try:
        p.mkdir(parents=True, exist_ok=True)
        t = p / ".writetest"
        t.write_text("ok", encoding="utf-8")
        t.unlink(missing_ok=True)
        return True
    except Exception:
        return False

if not _is_writable_dir(ARTIFACT_ROOT):
    raise RuntimeError(f"'outputs' is not writable: {ARTIFACT_ROOT.resolve()}")

TS = datetime.now().strftime("%Y%m%d-%H%M%S")
RUN_DIR = ARTIFACT_ROOT / f"{TS}_wiki_min"
RUN_DIR.mkdir(parents=True, exist_ok=True)

def save_text(name: str, content: str) -> str:
    p = RUN_DIR / f"{name}.md"
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(content, encoding="utf-8")
    return str(p)

def save_json(name: str, obj) -> str:
    p = RUN_DIR / f"{name}.json"
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps(obj, indent=2, ensure_ascii=False))
    return str(p)

print(f"[artifacts] Using: {RUN_DIR.resolve()}")


[artifacts] Using: /Users/jrizo/Documents/GitHub/jrizo/conv-rare-disease/src/outputs/20251119-125728_wiki_min


In [97]:
# --- Robust OpenAI caller with explicit error logging -------------------------
import os
from textwrap import dedent
from pathlib import Path
from datetime import datetime

# assumes you already defined RUN_DIR, save_text, save_json (from your "outputs" patch)
# if not, we create minimal fallbacks:
try:
    RUN_DIR
    save_text
    save_json
except NameError:
    ARTIFACT_ROOT = Path("outputs"); ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
    RUN_DIR = ARTIFACT_ROOT / f"{datetime.now().strftime('%Y%m%d-%H%M%S')}_openai_debug"
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    def save_text(name: str, content: str) -> str:
        p = RUN_DIR / f"{name}.md"; p.write_text(content, encoding="utf-8"); return str(p)
    def save_json(name: str, obj) -> str:
        p = RUN_DIR / f"{name}.json"; p.write_text(json.dumps(obj, indent=2, ensure_ascii=False)); return str(p)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
OPENAI_MODEL_PLAN = os.getenv("OPENAI_MODEL_PLAN", "gpt-4o-mini")
OPENAI_MODEL_SYN  = os.getenv("OPENAI_MODEL_SYN",  "gpt-4o-mini")
OPENAI_MODEL_FALLBACK = os.getenv("OPENAI_MODEL_FALLBACK", "gpt-4o")

# >>>> EDIT THESE THREE <<<<
DISEASE_NAME = "Duchenne Muscular Dystrophy"
AUDIENCE = "educated lay readers (patients/caregivers); secondary: clinicians/researchers"
STYLE = "concise, neutral, verifiable; Wikipedia-style; ~10th-grade reading level; define acronyms on first use; no medical advice"

def call_llm_or_emit_prompt(filename_prefix: str, prompt: str, system: str = "", *, stage: str = "plan") -> str:
    """
    OpenAI-only implementation with loud error reporting and a fallback model.
    Saves PROMPT and RESULT (or ERROR) into RUN_DIR.
    """
    model = OPENAI_MODEL_PLAN if stage == "plan" else OPENAI_MODEL_SYN

    if system:
        prompt_snapshot = f"[SYSTEM]\n{system}\n\n[USER]\n{prompt}"
    else:
        prompt_snapshot = prompt

    # No key? save prompt and stop
    if not OPENAI_API_KEY:
        save_text(f"{filename_prefix}_PROMPT", prompt_snapshot + "\n\n[Note] OPENAI_API_KEY not set; paste this into your LLM.")
        print("[llm] No OPENAI_API_KEY; wrote prompt file.")
        return ""

    # Try SDK v1 (new) first
    try:
        from openai import OpenAI
        client = OpenAI()  # new SDK reads key from env var
        resp = client.chat.completions.create(
            model=model,
            messages=(
                ([{"role": "system", "content": system}] if system else []) +
                [{"role": "user", "content": prompt}]
            ),
            temperature=0.2,
            max_tokens=2000,
        )
        out = resp.choices[0].message.content
        save_text(f"{filename_prefix}_PROMPT", prompt_snapshot)
        save_text(f"{filename_prefix}_RESULT", out)
        return out
    except Exception as e1:
        # Log and try fallback model
        err1 = f"[OpenAI primary model error] {type(e1).__name__}: {e1}"
        print(err1)

        try:
            from openai import OpenAI
            client = OpenAI()
            resp = client.chat.completions.create(
                model=OPENAI_MODEL_FALLBACK,
                messages=(
                    ([{"role": "system", "content": system}] if system else []) +
                    [{"role": "user", "content": prompt}]
                ),
                temperature=0.2,
                max_tokens=2000,
            )
            out = resp.choices[0].message.content
            save_text(f"{filename_prefix}_PROMPT", prompt_snapshot + f"\n\n[Note] Primary failed: {err1}\nUsed fallback: {OPENAI_MODEL_FALLBACK}")
            save_text(f"{filename_prefix}_RESULT", out)
            return out
        except Exception as e2:
            err2 = f"[OpenAI fallback error] {type(e2).__name__}: {e2}"
            print(err2)
            save_text(f"{filename_prefix}_PROMPT", prompt_snapshot + f"\n\n{err1}\n{err2}")
            save_text(f"{filename_prefix}_ERROR", err1 + "\n" + err2)
            return ""


In [76]:
# # --- Cell 1: Config + helpers (minimal, self-contained) -----------------------
# import os, json
# from pathlib import Path
# from datetime import datetime
# from textwrap import dedent

# # >>>> EDIT THESE THREE <<<<
# DISEASE_NAME = "Duchenne Muscular Dystrophy"
# AUDIENCE = "educated lay readers (patients/caregivers); secondary: clinicians/researchers"
# STYLE = "concise, neutral, verifiable; Wikipedia-style; ~10th-grade reading level; define acronyms on first use; no medical advice"

# # Output directory (consistent with earlier artifacts pattern)
# TS = datetime.now().strftime("%Y%m%d-%H%M%S")
# RUN_DIR = ARTIFACT_ROOT / f"{TS}_wiki_min"
# RUN_DIR.mkdir(parents=True, exist_ok=True)

# def save_text(name: str, content: str) -> str:
#     p = RUN_DIR / f"{name}.md"
#     p.write_text(content)
#     return str(p)

# def save_json(name: str, obj) -> str:
#     p = RUN_DIR / f"{name}.json"
#     p.write_text(json.dumps(obj, indent=2))
#     return str(p)

# # --- Vertex-aware LLM caller (replaces call_llm_or_emit_prompt) --------------
# import json
# from pathlib import Path
# from textwrap import dedent

# def call_llm_or_emit_prompt(filename_prefix: str, prompt: str, system: str = "") -> str:
#     """
#     Tries to call the configured LLM. Supports:
#       - Vertex (Gemini) via API key (google.generativeai)
#       - Vertex AI via service account / ADC (vertexai SDK)
#       - OpenAI (fallback if LLM_PROVIDER != 'vertex')
#     If no provider is ready, saves the prompt and returns "" so you can paste results manually.
#     """
#     from datetime import datetime
#     from pathlib import Path

#     # Save prompt helpers (assumes RUN_DIR is defined by earlier cells)
#     def _save(name, content):
#         p = RUN_DIR / f"{name}.md"
#         p.write_text(content)
#         return str(p)

#     merged_input = (system.strip() + "\n\n" + prompt.strip()) if system else prompt

#     # --- Vertex/Gemini path
#     if True:
#         try:
#             if GEMINI_API_KEY:
#                 print("Using Gemini API for LLM") 
#                 # Option A: Gemini API key (google.generativeai)
#                 import google.generativeai as genai
#                 genai.configure(api_key=GEMINI_API_KEY)
#                 model = genai.GenerativeModel(VERTEX_MODEL)
#                 # Passing a single merged string is fine for most use-cases
#                 resp = model.generate_content(merged_input)
#                 out = getattr(resp, "text", "") or ""
#                 _save(f"{filename_prefix}_PROMPT", merged_input)
#                 _save(f"{filename_prefix}_RESULT", out or "[EMPTY RESPONSE]")
#                 return out

#         except Exception as e:
#             _save(f"{filename_prefix}_PROMPT", merged_input + f"\n\n[Note: Vertex call error: {e}]")
#             return ""

#     # --- OpenAI fallback (only if you keep it around)
#     # --- If no provider is ready, emit prompt only
#     _save(f"{filename_prefix}_PROMPT", merged_input + "\n\n[Note: No LLM provider configured]")
#     return ""


In [103]:
# --- LLM connectivity + config debug -----------------------------------------
import os, socket, ssl, sys, json, time
from pathlib import Path

def _ping(host="api.openai.com", port=443, timeout=3):
    try:
        ctx = ssl.create_default_context()
        with socket.create_connection((host, port), timeout=timeout) as sock:
            with ctx.wrap_socket(sock, server_hostname=host) as ssock:
                return True
    except Exception as e:
        print(f"[connectivity] Cannot reach {host}:{port} -> {e}")
        return False

print("[env] OPENAI_API_KEY set? ", bool(os.getenv("OPENAI_API_KEY")))
print("[env] OPENAI_MODEL_PLAN: ", os.getenv("OPENAI_MODEL_PLAN", "gpt-4o-mini"))
print("[env] OPENAI_MODEL_SYN : ", os.getenv("OPENAI_MODEL_SYN",  "gpt-4o-mini"))

print("[connectivity] api.openai.com:443 reachable? ", _ping())

# Quick sanity on prompt objects if defined:
try:
    print("[vars] plan_prompt length:", len(plan_prompt))
except NameError:
    print("[vars] plan_prompt is not defined in this kernel.")
try:
    print("[vars] guideline_text length:", len(guideline_text))
except NameError:
    print("[vars] guideline_text is not defined in this kernel.")


[env] OPENAI_API_KEY set?  True
[env] OPENAI_MODEL_PLAN:  gpt-4o-mini
[env] OPENAI_MODEL_SYN :  gpt-4o-mini
[connectivity] api.openai.com:443 reachable?  True
[vars] plan_prompt length: 3402
[vars] guideline_text length: 2023


In [94]:
# --- Cell 2: Build the Wikipedia-style guideline from inputs ------------------
guideline_text = dedent(f"""
# Guideline for a Wikipedia-style Article
**Rare disease:** {DISEASE_NAME}  
**Audience:** {AUDIENCE}  
**Style:** {STYLE}

## Sections
1. Lead / Short summary (3–4 sentences; lay-accessible; key facts: cause, hallmark features, prevalence, inheritance, snapshot of treatment)
2. Overview / Definition
3. Signs and Symptoms / Clinical Features (onset, course, organ systems; phenotypic variants)
4. Genetics & Pathophysiology (gene(s), inheritance, penetrance, mechanisms; genotype–phenotype)
5. Diagnosis (clinical criteria, differentials, recommended tests; confirmatory molecular methods)
6. Management / Treatment (standard of care, supportive care, approved therapies; monitoring)
7. Prognosis / Natural History
8. Epidemiology (prevalence, founder effects, geography)
9. History / Nomenclature (discovery, synonyms, deprecated eponyms)
10. Research & Ongoing Trials (investigational therapies, notable studies)
11. See also
12. References (high-quality secondary sources; inline citations)
13. External links (major authorities only)

## Heuristics
- Prefer recent **reviews/guidelines** (≤10 years) over single case reports; if evidence is scarce, openly state uncertainty.
- Quantify findings (e.g., "~30%", "median age 7 y") with citations; avoid over-claiming.
- Clarify variants/synonyms; note deprecated eponyms.
- Consider tables for genotype–phenotype, differentials, and monitoring schedules.

## Preferred Sources (priority)
- **GeneReviews**, **NIH/NCATS GARD**, **Orphanet**, **MedlinePlus**
- Peer-reviewed **reviews** and **consensus/guidelines** (PubMed)
- **OMIM** (as a pointer, not the sole source)
- **ClinicalTrials.gov** (ongoing/interventional)
- **FDA/EMA** labels/approvals (if any)
- Major specialty societies (e.g., ACMG, AAN)

## Avoid
- Blogs, unsourced claims, social media, predatory journals.
""").strip()

guideline_path = save_text("guideline", guideline_text)



plan_prompt = dedent(f"""
Task: Create a step-by-step research plan to write a Wikipedia-style entry on **{DISEASE_NAME}** following the guideline below.

Deliverables: a numbered plan with milestones, search strings, inclusion/exclusion criteria, evidence grading, and a source list with rationale.
Constraints: neutral tone, verifiable citations only, prioritize secondary sources.

Guideline to follow:
{guideline_text}

Your plan must include:
1) Scope & key questions per section (diagnostic criteria? variants? management standards?).
2) Search strategy: PubMed queries (Boolean), Orphanet/GARD/GeneReviews pages, ClinicalTrials.gov filters, relevant society guidelines.
3) Screening rules: last 10 years for reviews when available; prefer meta-analyses/consensus; cohorts N≥10 when available; include case reports only if ultra-rare (mark low-N).
4) Evidence grading: guidelines > reviews > cohorts > case series > case reports.
5) Extraction schema: inheritance, gene, prevalence, hallmark signs, differentials, diagnostic tests, treatments, monitoring, outcomes.
6) Tables/figures planned: genotype–phenotype table; differential diagnosis table; therapy summary.
7) Risk & bias checks: outdated prevalence, small-N bias, founder effects, ascertainment bias; note controversies.
8) Milestones & outputs with timestamps (M1: sources assembled; M2: extraction complete; M3: draft; M4: fact-check pass).
""").strip()


In [98]:
# --- Cell 4: Synthesize the plan into a hierarchical research report ----------
# If plan_result is empty (no LLM), try to read from saved plan_RESULT.md (user may paste results)
plan_system = "You are a meticulous medical editor. Output a concrete, actionable plan with numbered steps and bullet points."
plan_result  = call_llm_or_emit_prompt("research_plan", plan_prompt, system=plan_system, stage="plan")

if not plan_result:
    print("meow")
    plan_result_path = RUN_DIR / "research_plan_RESULT.md"
    plan_result = plan_result_path.read_text() if plan_result_path.exists() else ""

synth_prompt = dedent(f"""
Task: Using the plan below, produce a hierarchical research report for **{DISEASE_NAME}** suitable to transform into a Wikipedia-style article.

Include: section headers, bullet-level facts with inline citations [Author Year] or [Database, Accessed YYYY-MM], and data tables.
Clearly separate **Facts**, **Uncertainties**, and **Controversies**. Do not invent citations—only use items listed.

Plan to execute:
{plan_result if plan_result else "[PASTE PLAN HERE]"}

Output structure (exactly):
- 1. Lead / Summary (4–6 bullets with citations)
- 2. Overview / Definition
- 3. Signs & Symptoms
- 4. Genetics & Pathophysiology
- 5. Diagnosis
- 6. Management / Treatment
- 7. Prognosis / Natural History
- 8. Epidemiology
- 9. History / Nomenclature
- 10. Research & Trials
- 11. Tables
  - T1. Genotype–Phenotype Mapping (Gene | Variant class | Key features | Penetrance | Source)
  - T2. Differential Diagnosis (Condition | Distinguishing features | Test | Source)
  - T3. Management Summary (Issue | Recommendation | Evidence level | Source)
- 12. Uncertainties & Controversies
- 13. References (grouped: Reviews/Guidelines; Databases; Clinical Studies; Regulatory/Trials)
""").strip()


# synth_system = "You are a senior biomedical writer. Follow the structure exactly. Use bullet points, table headers, and inline citations. No hallucinations."
# synth_result = call_llm_or_emit_prompt("synthesis", synth_prompt, system=synth_system)

# synth_result[:500] if synth_result else "Synthesis prompt saved for manual use."


In [102]:


rag_responses = []


literature_search_response: LiteratureSearchAgentResponse = await literature_search_agent.aforward(LiteratureSearchAgentRequest(topic=TOPIC, guideline=plan_result, with_synthesis=False))


with open(f"output/action_item_7_literature_search_response_{idx}.json", "w") as f:
    json.dump(literature_search_response.to_dict(), f, indent=2)
print(f"✅ Result saved to output/action_item_7_literature_search_response_{idx}.json")

rag_responses.extend(literature_search_response.rag_responses) # TODO: add the rag response to the list


Starting literature search for topic: Duchenne Muscular Dystrophy
Completeness check start.
Completeness check: False, reasoning: No dimensions of the topic have been explored, and initial foundational sections per guideline (overview, signs/symptoms, genetics/pathophysiology) are due as mandatory first steps.
Generated 3 next questions for exploration
Executing 3
RAG call start. Question: What are the key facts and a brief overview/definition of Duchenne Muscular Dystrophy (DMD)?. Question context: Establishes the foundational lead and overview section for the Wikipedia-style entry and is crucial for contextualizing all subsequent content.
RAG call start. Question: What are the hallmark signs, symptoms, and clinical features of DMD, including typical onset and organ systems affected?. Question context: Addresses the core clinical presentation necessary for readers to understand how DMD manifests and progresses, an essential section per guideline.
RAG call start. Question: What are the

python3.11(43445) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python3.11(43446) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python3.11(43447) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python3.11(43448) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python3.11(43449) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python3.11(43450) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


[FETCH]... ↓ https://my.clevelandclinic.org/health/diseases/23538-duchenne-muscular-dystrophy-dmd                 |
✓ | ⏱: 16.22s 

[SCRAPE].. ◆ https://my.clevelandclinic.org/health/diseases/23538-duchenne-muscular-dystrophy-dmd                 |
✓ | ⏱: 0.34s 

[COMPLETE] ● https://my.clevelandclinic.org/health/diseases/23538-duchenne-muscular-dystrophy-dmd                 |
✓ | ⏱: 16.60s 

...

[COMPLETE] ● https://synapse.patsnap.com/article/whats-the-la...al-trials-related-to-duchenne-muscular-dystrophy  |
✓ | ⏱: 45.22s 

[FETCH]... ↓ https://www.rarediseaseadvisor.com/hcp-resource/muscular-dystrophy-history/                          |
✓ | ⏱: 77.36s 

[SCRAPE].. ◆ https://www.rarediseaseadvisor.com/hcp-resource/muscular-dystrophy-history/                          |
✓ | ⏱: 0.16s 

[COMPLETE] ● https://www.rarediseaseadvisor.com/hcp-resource/muscular-dystrophy-history/                          |
✓ | ⏱: 77.52s 

Completed iteration 15, remaining budget: 0
Survey completed with 9 responses
✅ Result saved to output/action_item_7_literature_search_response_1.json


In [104]:
# Identify key insight for each rag response

# The goal of key insight identification is to identify the most important insight from the RAG response. Think of it as a one sentence summary of the RAG response that is most important to the question. The reason we do this is to help us reduce the noise in the rag responses and focus on the most important information when determing the final report outline.

class KeyInsightIdentifier(dspy.Signature):
    """
    You are GENERTING exactly ONE (1) KEY INSIGHT from the RAG repsonse. 
    It must be ONE sentence only, no more, no less. 

    REQURIEMENTS:
    - Directly answers the question, not vague.
    - Names actors and actions, incl time/place if avaible.
    - If there is a number/trend in the answer include it.
    - Keep inline [n] citations exactly as shown, do NOT change them.
    - No hedging like "maybe" or "could". No meta talk.
    - If no good fact is in the answer, return: "Insufficient evidence in retrieved sources to answer the question."
    """
    question: str = dspy.InputField(
        desc="The question that was asked"
    )
    question_context: str = dspy.InputField(
        desc="The context of the question"
    )
    answer: str = dspy.InputField(
        desc="The answer to the question aggregating information from external sources"
    )
    key_insight: str = dspy.OutputField(
        desc="One sentence key insight."
    )

key_insight_identifier = dspy.Predict(KeyInsightIdentifier)
key_insight_identifier_lm = init_lm(LanguageModelProviderConfig(
    provider=LanguageModelProvider.LANGUAGE_MODEL_PROVIDER_LITELLM_SERVER,
    model_name="gpt-4.1-mini", # NOTE: should we use a more powerful model? You can play around with it.
    temperature=1.0,
    max_tokens=10000,
    litellm_server_config=LiteLLMServerConfig(api_key=os.getenv("LITELLM_API_KEY"), api_base=os.getenv("LITELLM_API_BASE"))
))


for rag_response in tqdm(rag_responses):
    with dspy.context(lm=key_insight_identifier_lm):
        rag_response.key_insight = (await key_insight_identifier.aforward(
            question=rag_response.question,
            question_context=rag_response.question_context,
            answer=rag_response.answer
        )).key_insight

with open("output/action_item_7_rag_responses_with_key_insight.json", "w") as f:
    json.dump([rag_response.to_dict() for rag_response in rag_responses], f, indent=2)

for rag_response in rag_responses:
    print(rag_response.key_insight + "\n")
print(f"✅ Result saved to output/action_item_7_rag_responses_with_key_insight.json")

100%|█████████████████████████████████████████████| 9/9 [00:26<00:00,  2.98s/it]

Duchenne muscular dystrophy (DMD) is a severe X-linked recessive genetic disorder caused by mutations in the dystrophin gene, leading to progressive muscle weakness and degeneration primarily in males, with onset between ages 2 and 6 and affecting about 1 in 3,500 to 5,000 male births worldwide[1][2][3][4][5][6].

Duchenne muscular dystrophy (DMD) typically manifests between ages 2 and 6 years in boys, presenting with progressive symmetric proximal muscle weakness including difficulty running and climbing, characteristic calf pseudohypertrophy, use of Gowers’ sign, and involvement of cardiac, respiratory, gastrointestinal, and neurocognitive systems leading to loss of ambulation and life-threatening complications in adolescence[1][2][4][5][6][8].

Duchenne Muscular Dystrophy is an X-linked recessive disorder caused by mutations in the dystrophin gene on Xp21, primarily deletions (67.5%), leading to dystrophin deficiency that causes muscle membrane fragility, calcium influx, inflammatio

In [109]:
# Final report title and guideline proposal
# We will generate a TITLE + bullet GUIDELINE for the final rare-disease article.
# The guideline must mirror our target article structure.

class FinalWritingGuidelineProposal(dspy.Signature):
    f"""
    {synth_prompt}
    """

    # Inputs
    selected_theses: List[str] = dspy.InputField(
        desc="2–3 core theses about the disease (genetics, progression, therapy)."
    )
    key_insights: List[str] = dspy.InputField(
        desc="Key insights from RAG for this disease; may include diagnostics, therapy notes, or unmet needs."
    )

    # Outputs
    report_thesis: str = dspy.OutputField(
        desc="One headline-style title (8–14 words), disease-focused, no colon."
    )
    writing_guideline: str = dspy.OutputField(
        desc=(
            "Bullet-only guideline, ordered as: definition, genetics/etiology, clinical features, diagnosis, "
            "management/treatment, prognosis/impact, research/current trials, gaps/unknowns, safety note. "
            "Preserve any inline [n] citations from inputs."
        )
    )


# model init stays the same
final_writing_guideline_proposal = dspy.Predict(FinalWritingGuidelineProposal)
final_writing_guideline_proposal_lm = init_lm(LanguageModelProviderConfig(
    provider=LanguageModelProvider.LANGUAGE_MODEL_PROVIDER_LITELLM_SERVER,
    model_name="gpt-4.1",
    temperature=1.0,
    max_tokens=10000,
    litellm_server_config=LiteLLMServerConfig(
        api_key=os.getenv("LITELLM_API_KEY"),
        api_base=os.getenv("LITELLM_API_BASE")
    )
))

with dspy.context(lm=final_writing_guideline_proposal_lm):
    final_writing_guideline_proposal_response = await final_writing_guideline_proposal.aforward(
        selected_theses=selected_theses,
        key_insights=[rr.key_insight for rr in rag_responses],
    )

final_writing_thesis = final_writing_guideline_proposal_response.report_thesis
final_writing_guideline = final_writing_guideline_proposal_response.writing_guideline

with open("output/action_item_7_final_writing_guideline_proposal.json", "w") as f:
    json.dump({
        "report_thesis": final_writing_thesis,
        "writing_guideline": final_writing_guideline
    }, f, indent=2)

print(f"report title: {final_writing_thesis}\n")
print(f"writing guideline:\n{final_writing_guideline}")
print("✅ Result saved to output/action_item_7_final_writing_guideline_proposal.json")


report title: Genetic mutations in the dystrophin gene drive Duchenne Muscular Dystrophy progression and highlight urgent therapeutic needs

writing guideline:
- Define Duchenne Muscular Dystrophy (DMD) as a severe, progressive neuromuscular disease affecting primarily males, with early childhood onset and global rarity[1][2][3][4][5][6].
- Explain genetics/etiology: X-linked recessive inheritance from mutations (primarily deletions/duplications) in the dystrophin gene on Xp21; these mutations result in dystrophin deficiency responsible for muscle membrane instability, degeneration, and multisystem involvement; nearly all cases confirm DMD gene abnormality, supporting molecular diagnostic protocols (Emery, 2023)[1][2][4][7][8].
- Summarize clinical features: Onset ages 2–6; progressive symmetric proximal muscle weakness, calf pseudohypertrophy, Gowers’ sign, delayed motor milestones, loss of ambulation in adolescence; complications include cardiomyopathy, respiratory insufficiency, gas

In [110]:
def _normalize_rag_response_citation_indices(rag_responses: List[RagResponse]) -> Tuple[List[str], List[RetrievedDocument]]:
        f"""
        {guideline_large}
        Normalize citation indices across multiple RAG (retrieval-augmented generation) responses.

        Each `RagResponse` contains:
        - `answer`: a string with inline citations like [1], [2], ...
        - `cited_documents`: the list of documents those citations refer to

        Problem:
        Citation indices restart at [1] for every response, but when combining answers,
        we want all citations to point to a single global list of retrieved documents.

        What this function does:
        1. Iterates over all RAG responses in order.
        2. Shifts the local citation indices in each answer so that they correctly map
            into the combined list of all retrieved documents.
            - For example, if the first response cited 3 docs ([1], [2], [3]),
            then the second response’s citations start at [4], not [1].
        3. Prefixes each updated answer with its corresponding sub-question for clarity.
        4. Returns:
            - A list of normalized answers (with corrected citation indices).
            - The flattened list of all retrieved documents in the proper order.

        Example:
            Input (two RAG responses):
                R1: "Paris is in France [1].", docs=[docA]
                R2: "Berlin is in Germany [1].", docs=[docB]

            Output:
                answers = [
                "Sub-question: ...\nAnswer: Paris is in France [1].",
                "Sub-question: ...\nAnswer: Berlin is in Germany [2]."
                ]
                documents = [docA, docB]
        """
        all_documents: List[RetrievedDocument] = []
        all_updated_answers: List[str] = []
        for idx, rag_response in enumerate(rag_responses):
            citation_offset = len(all_documents)
            updated_answer = rag_response.answer
            for i in range(len(rag_response.cited_documents)):
                updated_answer = updated_answer.replace(f"[{i+1}]", f"[tmp_{citation_offset+i+1}]")
            for i in range(len(rag_response.cited_documents)):
                updated_answer = updated_answer.replace(f"[tmp_{citation_offset+i+1}]", f"[{citation_offset+i+1}]")

            all_updated_answers.append(
                f"Sub-question: {rag_response.question}\nAnswer: {updated_answer}\n")
            all_documents.extend(rag_response.cited_documents)
        return all_updated_answers, all_documents

In [111]:
# Final report synthesis
# We will synthesize the final rare-disease article using the generated title, guideline, and RAG responses.

class FinalReportSynthesizer(dspy.Signature):
    """
    You are composing a Wikipedia-style encyclopedia article (patient-accessible, neutral tone)
from a **writing guideline** and **gathered information** (RAG answers with pre-normalized
numeric citations).

PIPELINE CONTEXT (DO NOT IGNORE)
- Section scaffold: May derive from Wikipedia section mining + internal guideline fusion.
- Scope: Rare/orphan disease; audience is educated lay readers first, clinicians second.
- Sources: Only use facts present in `gathered_information`. Numeric citations have already
  been normalized by the pipeline. You must reuse those numbers exactly (see Citation Policy).
- Do NOT add external knowledge. If something is missing, state so briefly.

PROSE-FIRST STYLE
- Lead: 2–4 sentences summarizing definition, hallmark systems, inheritance, and high-level care.
- Sections: write paragraphs with topic sentences and transitions (“However,” “Additionally,” “In contrast,”).
- Bullets: allowed for short enumerations only (e.g., symptom lists); ≤15% of total lines.
- No second person; no advice/imperatives.

HIERARCHY & ORDER
- Use ## for main sections; ### for subsections. Avoid deeper nesting.
- Follow the exact order provided by `writing_guideline`. If a “What Is Not Known”/“Uncertainties”
  section exists, keep it near the end.

CITATION POLICY (STRICT)
1) Allowed format
   - Only numeric inline citations: [1], [2], [3], …  No URLs, domains, or alphanumeric IDs in brackets.
2) Source of truth
   - Reuse citation numbers exactly as they appear in `gathered_information`. Do not renumber.
3) Placement
   - Attach immediately after the fact/number/name it supports. Example:
     “DMD is caused by mutations in the DMD gene on Xp21.2.[1]”
4) Count per fact
   - Default 1 number. Use 2 only if the sentence genuinely combines two distinct supported claims
     or you’re showing agreement across guidelines. Max 3.
   - If inputs showed long chains like [1][2][13][30], shorten to the 1–2 most relevant.
5) Duplicates
   - If two sentences cite the same number for the same claim, you may keep the first and avoid repeats.
6) Malformed/non-numeric citations in inputs
   - Replace only if a numeric ID for the same claim exists in the same input block; otherwise omit the citation
     and keep the sentence only if it’s supported elsewhere in the same paragraph.
7) Ascending order
   - When multiple citations appear in one sentence, sort ascending: “[1][5]”, not “[5][1]”.
8) Missing support
   - If no numeric citation in `gathered_information` supports a sentence, remove the sentence.
9) Section intros
   - For a generic section intro summarizing multiple cited claims, use the earliest applicable number
     from that section’s inputs (e.g., “[23]” or “[23][24]”).
10) No cross-section recycling
   - Don’t copy a number from therapies into genetics unless that source supported genetics.

MISSING OR LIMITED EVIDENCE
- If a section lacks support in the inputs, include a one-line note:
  “Limited information was available in the retrieved sources.” (no citation).

SAFETY NOTE
- Immediately after the H1 title, include:
  “This article is for education only and does not replace advice from your clinician.”
    """

    report_thesis: str = dspy.InputField(
        desc="The article/report title that should appear as H1."
    )
    writing_guideline: str = dspy.InputField(
        desc="Bullet-format guideline listing the sections and key points, in order."
    )
    gathered_information: str = dspy.InputField(
        desc="All RAG answers with normalized inline citations, newline-separated."
    )
    report_style: str = dspy.InputField(
        desc="Optional style hint, e.g. 'patient-friendly, concise, no medical advice'.",
        default="patient-friendly, concise, no medical advice"
    )

    final_report: str = dspy.OutputField(
        desc="The final rare-disease article in markdown, with H1 title, safety note, and ordered sections."
    )


final_report_synthesizer = dspy.Predict(FinalReportSynthesizer)
final_report_synthesizer_lm = init_lm(LanguageModelProviderConfig(
    provider=LanguageModelProvider.LANGUAGE_MODEL_PROVIDER_LITELLM_SERVER,
    model_name="gpt-5-mini",
    temperature=1.0,
    max_tokens=20000,
    litellm_server_config=LiteLLMServerConfig(
        api_key=os.getenv("LITELLM_API_KEY"),
        api_base=os.getenv("LITELLM_API_BASE")
    )
))

# normalize citations from RAG (your existing helper)
all_updated_answers, all_documents = _normalize_rag_response_citation_indices(rag_responses)
gathered_information = "\n".join(all_updated_answers)

with dspy.context(lm=final_report_synthesizer_lm):
    final_report = (await final_report_synthesizer.aforward(
        report_thesis=final_writing_thesis,
        writing_guideline=final_writing_guideline,
        gathered_information=gathered_information,
        report_style="patient-friendly, preserve section order, no medical advice"
    )).final_report

with open("output/action_item_7_final_report_raw.md", "w") as f:
    f.write(final_report)
print("✅ Result saved to output/action_item_7_final_report_raw.md")


✅ Result saved to output/action_item_7_final_report_raw.md


In [23]:
# TODO: manually add bibliography to the final report. Review the output. Adjust the prompt and rerun the report synthesis if necessary. Make sure it has title, executive summary, sections with desired inline citations, and bibliography.

bibliography = "# Bibliography \n\n"
for index, doc in enumerate(all_documents, 1):
    bibliography += f"{index}. **{doc.title}**. Available at: {doc.url}\n"

final_report_with_bibliography = final_report + "\n\n" + bibliography

with open("output/action_item_7_final_report.md", "w") as f:
    f.write(final_report_with_bibliography)
print(f"✅ Result saved to output/action_item_7_final_report.md")

✅ Result saved to output/action_item_7_final_report.md


In [20]:
# API endpoint URL
api_url = "https://cs224v-database-agent.genie.stanford.edu/database-exploration"

# Prepare the request payload
payload = {
    "topic": TOPIC,
    "seed_questions": seed_questions,
    "lm_config": database_exploration_lm_config.to_dict()
}

# NOTE: uncomment the code below to make the request

# print("Making request to database exploration endpoint... Might take up to 30 minutes")
# async with httpx.AsyncClient(timeout=6000.0) as client:
#     r = await client.post(api_url, json=payload)
#     r.raise_for_status()
#     database_exploration_response = r.json()

# with open("output/action_item_5_database_exploration.json", "w") as f:
#     json.dump(database_exploration_response, f, indent=2)

# print(f"✅ Result saved to output/action_item_5_database_exploration.json")


NameError: name 'seed_questions' is not defined